# Imports

In [44]:
#Add utils directory to system path, otherwise we get a ModuleNotFoundError (utils is in different directory than execute)
import sys
import os

current_dir = os.getcwd()
notebooks_dir = os.path.abspath(os.path.join(current_dir, "..", ".."))
if notebooks_dir not in sys.path:
    sys.path.append(notebooks_dir)

In [45]:
#Imports
import pandas as pd
import warnings
import matplotlib.pyplot as plt
from notebooks.utils.util_dataprep import Util_DataPrep
from notebooks.utils.util_training import Util_Training
from notebooks.utils.util_hyperparameter import Util_Hyperparameter
from notebooks.utils.transformer import Transformer, HyperparameterTuningTransformer
import pickle
import json

In [46]:
%load_ext autoreload
%autoreload 2
warnings.filterwarnings("ignore")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Read in country data + Instantiate Util classes

In [47]:
#Define the file paths
filename = 'countries_data.xlsx'
estimates_path = '../Electricity_point_estimates_december_utygi.json'

#Read the country data (entire Excel file) into a dictionary of DataFrames
#This excel sheet was created by the merge_datasets.ipynb notebook
countries_dict = pd.read_excel(filename, sheet_name=None, index_col=0)

#Instantiate the Util classes
u_prep = Util_DataPrep()
u_training = Util_Training()
u_transformer = Transformer()
u_transformer_hyperparameters = HyperparameterTuningTransformer()
u_hyperparameter = Util_Hyperparameter()

In [48]:
#Define reverse mapping to write the results for the different countries (e.g. Austria) to
#the Electricity_point_estimates json at the correct position (e.g. AT)
country_mapping_reversed = {
    'Austria': 'AT',
    'Belgium': 'BE',
    'Bulgaria': 'BG',
    'Cyprus': 'CY',
    'Czechia': 'CZ',
    'Germany': 'DE',
    'Denmark': 'DK',
    'Estonia': 'EE',
    'Greece': 'EL',
    'Spain': 'ES',
    'Finland': 'FI',
    'France': 'FR',
    'Croatia': 'HR',
    'Hungary': 'HU',
    'Ireland': 'IE',
    'Italy': 'IT',
    'Lithuania': 'LT',
    'Luxembourg': 'LU',
    'Latvia': 'LV',
    'Malta': 'MT',
    'Netherlands': 'NL',
    'Poland': 'PL',
    'Portugal': 'PT',
    'Romania': 'RO',
    'Sweden': 'SE',
    'Slovenia': 'SI',
    'Slovakia': 'SK'
}

In [49]:
#Define X and y columns
X_cols = [
    'month',
    'year',
    'AirTemperature',
    'Distribution_losses',
    'ElectricityDemand',
    'Exports',
    'Exports_to_EU',
    'GlobalHorizontalIrradiance',
    'Hydropower(Reservoir)',
    'Hydropower(RunOfRiver)',
    'Imports',
    'Imports_from_EU',
    'MeanSeaLevelPressure',
    'SolarPVPower',
    'TotalPrecipitation',
    'Transformation_Input',
    'WindPowerOnshore',
    'WindSpeed'
]

y_col = "Available_to_internal_market"

# Fill missing values at the end
## Approach : Data shifting

In [50]:
#Function to define the order of preprocessing operations
def prepData(country_name):
        
        #Get the dict with all data of one country
        country_df = countries_dict.get(country_name)

        #202 = November 2024, 203 = Dezember 2024
        month_to_predict_idx = 203

        #Get the number of missing months in the data for all X columns
        columns_with_missing_shift = u_prep.getColumnsWithNumberOfMissingMonth(country_df, month_to_predict_idx, y_col)

        #Approach: Shift columns with missing values forward to the index of the current month (e.g. December 2024 == 204)
        shifted_df = u_prep.shift_columns(country_df, columns_with_missing_shift)

        #Decide whether to use all data or only from a specific period
        #shifted_df = shifted_df[shifted_df['year'] >= 2017]

        #Get number of missing targets (Available to internal market)
        #This equals the number of predictions which have to be made
        number_of_missing_targets = u_prep.getNumberOfMissingTargets(shifted_df, month_to_predict_idx, y_col)

        #For training purpose -> remove the last 4 rows because of missing y and split the data in train / test
        X_train, y_train, X_test, y_test = u_prep.remove_data_and_split(shifted_df, number_of_missing_targets, X_cols, y_col)

        return X_train, y_train, X_test, y_test, number_of_missing_targets

# Fit models

## Load tuned Hyperparameters

In [51]:
with open("../hyperparameter/sarima_hyperparameter/sarima_model_order.pkl", "rb") as file:
        sarima_model_order_dict = pickle.load(file)

with open("../hyperparameter/sarima_hyperparameter/sarima_seasonal_order.pkl", "rb") as file:
        sarima_seasonal_order_dict = pickle.load(file)

with open("../hyperparameter/best_transformer_hyperparameters.pkl", "rb") as file:
        transformer_best_hyperparameters = pickle.load(file)

with open("../hyperparameter/best_xgb_hyperparameters.pkl", "rb") as file:
        xgb_best_hyperparameters = pickle.load(file)

with open("../hyperparameter/best_rf_hyperparameters.pkl", "rb") as file:
        rf_best_hyperparameters = pickle.load(file)

with open("../hyperparameter/best_ridge_hyperparameters.pkl", "rb") as file:
        ridge_best_hyperparameters = pickle.load(file)

## Train models and predict

In [52]:

#Define dicts to store the results
ensemble_msre_dict1 = {}
ensemble_msre_dict2 = {}
ensemble_msre_dict3 = {}
ensemble_msre_dict4 = {}
sarima_msre_dict = {}
xgb_msre_dict = {}
transformer_msre_dict = {}
rf_msre_dict = {}
ridge_msre_dict = {}
lin_msre_dict = {}

#Apply all models to the data of each country
for country,df in countries_dict.items():

    print(f"Processing country: {country}")

    #Call the data preprocessing method defined above, test only with the last values to predict
    X_train, y_train, X_test, y_test, number_of_missing_targets = prepData(country)
    y_test = y_test.iloc[-1]

    #Train and predict with sarima
    y_pred_sarima = u_training.train_sarima_and_predict(y_train,sarima_model_order_dict[country],
                                                        sarima_seasonal_order_dict[country],
                                                        number_of_missing_targets)[-1]
    sarima_msre = u_training.calculate_msre(y_test, y_pred_sarima)
    sarima_msre_dict[country] = sarima_msre

    #Train and predict with xgboost
    y_pred_xgb = u_training.train_xgb_and_predict(X_train, y_train, X_test, xgb_best_hyperparameters[country])[-1]
    xgb_msre = u_training.calculate_msre(y_test, y_pred_xgb)
    xgb_msre_dict[country] = xgb_msre

    #Train and predict with linear
    y_pred_lin = u_training.train_lin_reg_and_predict(X_train, y_train, X_test)[-1]
    lin_msre = u_training.calculate_msre(y_test, y_pred_lin)
    lin_msre_dict[country] = lin_msre

    #Example code for Hyperparameter Search for Transformer
    #best_params = u_transformer_hyperparameters.perform_hyperparameter_search(X_train, y_train)
    #transformer_best_hyperparameters[country] = best_params

    #Train and predict with Transformer
    model, scaler_X, scaler_y, initial_sequence, seq_size = u_transformer.train_transformer(X_train, y_train, transformer_best_hyperparameters[country])
    y_pred_transformer = u_transformer.predict_next_values(model, scaler_X, scaler_y, initial_sequence, seq_size, X_test, number_of_missing_targets)[-1]
    transformer_msre = u_training.calculate_msre(y_test, y_pred_transformer)
    transformer_msre_dict[country] = transformer_msre

    #Train and predict with Random Forest
    y_pred_rf = u_training.train_random_forest_and_predict(X_train, y_train, X_test, rf_best_hyperparameters[country])[-1]
    rf_msre = u_training.calculate_msre(y_test, y_pred_rf)
    rf_msre_dict[country] = rf_msre

    #Train and predict with Ridge
    y_pred_ridge = u_training.train_ridge_and_predict(X_train, y_train, X_test, ridge_best_hyperparameters[country])[-1]
    ridge_msre = u_training.calculate_msre(y_test, y_pred_ridge)
    ridge_msre_dict[country] = ridge_msre
    
    #Use this code if you want to evaluate which models work the best in an ensemble
    # models = {
    #     "Sarima": y_pred_sarima,
    #     "Transformer": y_pred_transformer,
    #     "XGB": y_pred_xgb,
    #     "Ridge": y_pred_ridge,
    #     "Random Forest": y_pred_rf,
    #     "Linear": y_pred_lin
    # }
    # ensemble_msre_dict[country] = u_training.evaluateDifferentEnsembles(models, y_test)

    #Assume you have found good ensembles, combine their predictions here:
    #Predict with ensemble
    list_of_predictions1 = [y_pred_sarima, y_pred_xgb, y_pred_rf, y_pred_ridge]
    list_of_predictions2 = [y_pred_sarima, y_pred_xgb, y_pred_rf]
    list_of_predictions3 = [y_pred_sarima, y_pred_transformer, y_pred_ridge]
    list_of_predictions4 = [y_pred_sarima, y_pred_transformer, y_pred_xgb, y_pred_rf, y_pred_ridge, y_pred_lin]

    y_pred_ensemble1 = u_training.make_ensemble_prediction(list_of_predictions1)
    y_pred_ensemble2 = u_training.make_ensemble_prediction(list_of_predictions2)
    y_pred_ensemble3 = u_training.make_ensemble_prediction(list_of_predictions3)
    y_pred_ensemble4 = u_training.make_ensemble_prediction(list_of_predictions4)

    ensemble_msre1 = u_training.calculate_msre(y_test, y_pred_ensemble1)
    ensemble_msre_dict1[country] = ensemble_msre1
    ensemble_msre2 = u_training.calculate_msre(y_test, y_pred_ensemble2)
    ensemble_msre_dict2[country] = ensemble_msre2
    ensemble_msre3 = u_training.calculate_msre(y_test, y_pred_ensemble3)
    ensemble_msre_dict3[country] = ensemble_msre3
    ensemble_msre4 = u_training.calculate_msre(y_test, y_pred_ensemble4)
    ensemble_msre_dict4[country] = ensemble_msre4

Processing country: Austria
Processing country: Belgium
Processing country: Bulgaria
Processing country: Cyprus
Processing country: Czechia
Processing country: Germany
Processing country: Denmark
Processing country: Estonia
Processing country: Greece
Processing country: Spain
Processing country: Finland
Processing country: France
Processing country: Croatia
Processing country: Hungary
Processing country: Ireland
Processing country: Italy
Processing country: Lithuania
Processing country: Luxembourg
Processing country: Latvia
Processing country: Netherlands
Processing country: Poland
Processing country: Portugal
Processing country: Romania
Processing country: Sweden
Processing country: Slovenia
Processing country: Slovakia


# Multiply with weights and evaluate

In [53]:
electricity_weights = pd.read_csv('../../data/electricity_weights.csv', sep=";", index_col=0)
electricity_weights['Country Name'] = electricity_weights['Country Name'].replace('Czech Republic', 'Czechia')
electricity_weights.head()

,Weights,ISO2,Country Name
0,1.93,AT,Austria
1,1.78,BE,Belgium
2,1.43,BG,Bulgaria
3,1.49,CY,Cyprus
4,1.95,CZ,Czechia


In [54]:
for country in countries_dict.keys():
    #Get the weight for the current country from the weights DataFrame
    weight = electricity_weights.loc[electricity_weights['Country Name'] == country, 'Weights'].values[0]
    
    #Multiply each MSRE value by the country's weight
    ensemble_msre_dict1[country] *= weight
    ensemble_msre_dict2[country] *= weight
    ensemble_msre_dict3[country] *= weight
    ensemble_msre_dict4[country] *= weight
    sarima_msre_dict[country] *= weight
    transformer_msre_dict[country] *= weight
    xgb_msre_dict[country] *= weight
    lin_msre_dict[country] *= weight

In [ ]:
msre_dicts = {
    "ensemble1": ensemble_msre_dict1,
    "ensemble2": ensemble_msre_dict2,
    "ensemble3": ensemble_msre_dict3,
    "ensemble4": ensemble_msre_dict4,
    "sarima": sarima_msre_dict,
    "transformer": transformer_msre_dict,
    "xgboost": xgb_msre_dict,
    "linear_regression": lin_msre_dict
}

sum_of_values_dict = {}

for model_name, msre_dict in msre_dicts.items():
    #Sort the dictionary by values and select the five lowest values
    lowest_values_sum = sum(dict(sorted(msre_dict.items(), key=lambda item: item[1])[:5]).values())
    sum_of_values_dict[model_name] = lowest_values_sum
    print(f"Sum of the 5 lowest MSRE values for {model_name}: {lowest_values_sum}")

Sum of the 5 lowest MSRE values for ensemble1: 0.0006239851121762664
Sum of the 5 lowest MSRE values for ensemble2: 7.292052067290215e-05
Sum of the 5 lowest MSRE values for ensemble3: 0.0009271172775835939
Sum of the 5 lowest MSRE values for ensemble4: 0.0009324863746574066
Sum of the 5 lowest MSRE values for sarima: 0.00010534545359350736
Sum of the 5 lowest MSRE values for transformer: 0.005838046566279167
Sum of the 5 lowest MSRE values for xgboost: 0.0004115893427241984
Sum of the 5 lowest MSRE values for linear_regression: 0.0012652502328807822
